# Experiment 2: Counterfactual Sensitivity

Family-level analysis: fragility rate, severity distribution, pattern analysis.

**Primary analysis unit**: family (not instance). Family members are correlated, not IID.

In [9]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
from utils import load_all_results

from bcbench.analysis.aggregator import build_families
from bcbench.analysis.family import FamilyOutcome
from bcbench.analysis.metrics import family_type_distribution, fragility_rate, mean_severity
from bcbench.results.base import ExecutionBasedEvaluationResult

# Claude Opus 4.6 has both base + CF results for full family analysis
SETUP = "copilot-cf-opus-4-6"
MODEL_LABEL = "Claude Opus 4.6"

all_results = load_all_results(category="bug-fix")


def df_to_results(df: pd.DataFrame) -> list[ExecutionBasedEvaluationResult]:
    results = []
    for _, row in df.iterrows():
        results.append(
            ExecutionBasedEvaluationResult(
                instance_id=row["instance_id"],
                project=row.get("project", ""),
                model=row.get("model", "unknown"),
                agent_name="GitHub Copilot",
                category="cf",
                resolved=bool(row.get("resolved", False)),
                build=bool(row.get("build", False)),
                output=row.get("output", ""),
            )
        )
    return results


df = all_results[SETUP]
results = df_to_results(df)
n_base = sum(1 for r in results if "__cf-" not in r.instance_id)
n_cf = sum(1 for r in results if "__cf-" in r.instance_id)
families = build_families(results)

families_by_model = {MODEL_LABEL: families}

print(f"{MODEL_LABEL}: {len(results)} results (base={n_base}, cf={n_cf}) -> {len(families)} families")
print(f"Base results come from 5 runs; CF results from run-25281894565")

Claude Opus 4.6: 626 results (base=505, cf=121) -> 71 families
Base results come from 5 runs; CF results from run-25281894565


## Family Outcome Table

Each row = one family. Columns: family_id, layer, base, cf1..cfN, pattern, type, fragile, severity.

In [10]:
def families_to_df(families: list[FamilyOutcome]) -> pd.DataFrame:
    rows = []
    for f in families:
        row = {
            "family_id": f.family_id,
            "layer": f.failure_layer.value if f.failure_layer else "unclassified",
            "base": int(f.base.passed),
            "pattern": str(f.pattern),
            "type": f.family_type.value,
            "fragile": int(f.is_fragile),
            "severity": f.severity,
            "cf_total": f.cf_total,
            "cf_fail_count": f.cf_fail_count,
        }
        for i, cf in enumerate(f.cfs):
            row[f"cf{i + 1}"] = int(cf.passed)
        rows.append(row)
    return pd.DataFrame(rows)


print("Family outcome table builder ready. Load results to populate.")

Family outcome table builder ready. Load results to populate.


## Family Type Distribution

In [11]:
def plot_family_type_distribution(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        dist = family_type_distribution(families)
        total = sum(dist.values())
        for ftype, count in dist.items():
            rows.append(
                {
                    "Model": model,
                    "Family Type": ftype,
                    "Count": count,
                    "Proportion (%)": round(count / total * 100, 1) if total else 0,
                }
            )

    df = pd.DataFrame(rows)
    fig = px.bar(
        df,
        x="Model",
        y="Proportion (%)",
        color="Family Type",
        title="Family Type Distribution by Model",
        barmode="stack",
        text_auto=True,
        color_discrete_map={
            "stable-correct": "#2ecc71",
            "fragile": "#e74c3c",
            "unsolved": "#95a5a6",
            "inconsistent": "#f39c12",
        },
    )
    fig.show()
    return df


print("plot_family_type_distribution() ready")

plot_family_type_distribution() ready


## Fragility Rate (Main Thesis Figure)

In [12]:
def plot_fragility_rate(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        rate = fragility_rate(families)
        sev = mean_severity(families)
        rows.append(
            {
                "Model": model,
                "Fragility Rate (%)": round(rate * 100, 1),
                "Mean Severity": round(sev, 3) if sev is not None else None,
            }
        )

    df = pd.DataFrame(rows)
    fig = px.bar(
        df,
        x="Model",
        y="Fragility Rate (%)",
        title="Family Fragility Rate: P(CF fail | Base correct)",
        text_auto=True,
        color_discrete_sequence=["#e74c3c"],
    )
    fig.update_layout(yaxis_range=[0, 100])
    fig.show()
    return df


print("plot_fragility_rate() ready")

plot_fragility_rate() ready


## Severity Distribution

In [13]:
def plot_severity_distribution(families_by_model: dict[str, list[FamilyOutcome]]):
    rows = []
    for model, families in families_by_model.items():
        for f in families:
            if f.severity is not None:
                rows.append({"Model": model, "Severity": f.severity, "Type": f.family_type.value})

    df = pd.DataFrame(rows)
    if not df.empty:
        fig = px.box(
            df,
            x="Model",
            y="Severity",
            color="Model",
            title="Fragility Severity Distribution (N_cf_fail / N_cf)",
            points="all",
        )
        fig.show()
    return df


print("plot_severity_distribution() ready")

plot_severity_distribution() ready


## Pattern Analysis

In [14]:
from collections import Counter


def analyze_patterns(families: list[FamilyOutcome]) -> pd.DataFrame:
    pattern_counts = Counter(f.pattern for f in families)
    rows = [{"Pattern": str(p), "Count": c, "Pct (%)": round(c / len(families) * 100, 1)} for p, c in pattern_counts.most_common()]
    return pd.DataFrame(rows)


print("analyze_patterns() ready")

analyze_patterns() ready


## Run All Analysis

In [15]:
# Family outcome table
for model, families in families_by_model.items():
    print(f"\n=== {model} ({len(families)} families) ===")
    df = families_to_df(families)
    display(df)

# Family type distribution
type_df = plot_family_type_distribution(families_by_model)
display(type_df)

# Fragility rate
frag_df = plot_fragility_rate(families_by_model)
display(frag_df)

# Severity distribution
sev_df = plot_severity_distribution(families_by_model)

# Pattern analysis
for model, families in families_by_model.items():
    print(f"\n=== {model} patterns ===")
    pat_df = analyze_patterns(families)
    display(pat_df)


=== Claude Opus 4.6 (71 families) ===


,family_id,layer,base,pattern,type,fragile,severity,cf_total,cf_fail_count,cf1,cf2,cf3
0,microsoftInternal__NAV-174087,unclassified,1,"(1, 1, 1)",stable-correct,0,0.000000,2,0,1,1.0,NaN
1,microsoftInternal__NAV-174794,unclassified,1,"(1, 1, 0, 0)",fragile,1,0.666667,3,2,1,0.0,0.0
2,microsoftInternal__NAV-175765,unclassified,1,"(1, 0, 0, 0)",fragile,1,1.000000,3,3,0,0.0,0.0
3,microsoftInternal__NAV-176082,unclassified,1,"(1, 0, 0, 1)",fragile,1,0.666667,3,2,0,0.0,1.0
4,microsoftInternal__NAV-176150,unclassified,1,"(1, 0, 1)",fragile,1,0.500000,2,1,0,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
66,microsoftInternal__NAV-227219,unclassified,1,"(1, 1)",stable-correct,0,0.000000,1,0,1,NaN,NaN
67,microsoftInternal__NAV-227358,unclassified,1,"(1, 1, 1)",stable-correct,0,0.000000,2,0,1,1.0,NaN
68,microsoft__BCApps-4699,unclassified,1,"(1, 0)",fragile,1,1.000000,1,1,0,NaN,NaN
69,microsoft__BCApps-4822,unclassified,1,"(1, 1, 1)",stable-correct,0,0.000000,2,0,1,1.0,NaN


,Model,Family Type,Count,Proportion (%)
0,Claude Opus 4.6,stable-correct,35,49.3
1,Claude Opus 4.6,fragile,15,21.1
2,Claude Opus 4.6,unsolved,14,19.7
3,Claude Opus 4.6,inconsistent,7,9.9


,Model,Fragility Rate (%),Mean Severity
0,Claude Opus 4.6,30.0,0.24



=== Claude Opus 4.6 patterns ===


,Pattern,Count,Pct (%)
0,"(1, 1)",16,22.5
1,"(1, 1, 1)",13,18.3
2,"(0, 0)",8,11.3
3,"(1, 0)",6,8.5
4,"(1, 1, 1, 1)",6,8.5
5,"(0, 0, 0)",4,5.6
6,"(0, 1)",4,5.6
7,"(0, 0, 0, 0)",2,2.8
8,"(1, 0, 0)",2,2.8
9,"(1, 1, 0, 0)",1,1.4
